In [0]:
from pyspark.sql.functions import col,to_date

In [0]:

# Creating source Data
data = [
    (3,"g","2025-01-05"),
    (4,"d","2025-01-06"),
    (5,"e","2025-01-06"),
    (2,"h","2025-01-06"),
    (1,"i","2025-01-07"),
    (6,"f","2025-01-08"),

]

schema=["id","name","updated_date"]

source=spark.createDataFrame(data,schema)
source=source.withColumn("updated_date",to_date(col("updated_date")))
display(source)
source.write.mode("overwrite").saveAsTable("practice.default.source")

In [0]:
data = [
    (1,"practice.default.target",3),
]

schema=["table_id","table_name","max_key"]

config_table=spark.createDataFrame(data,schema)

display(config_table)

config_table.write.mode("overwrite").saveAsTable("practice.default.config_table")

In [0]:
%sql
-- config table for storing max available suugate key
select * from practice.default.config_table

In [0]:
data = [
    (1,1,"a","2025-01-01","2099-01-01","y"),
    (2,2,"b","2025-01-01","2099-01-01","y"),
    (3,3,"c","2025-01-01","2099-01-01","y"),
]

schema=["sk","id","name","effective_date","end_date","valid_flag"]

target=spark.createDataFrame(data,schema)
target=target.withColumn("effective_date",to_date(col("effective_date")))
target=target.withColumn("end_date",to_date(col("end_date")))
display(target)
target.write.mode("overwrite").saveAsTable("practice.default.target")

In [0]:
%sql
-- Data at Source table
select * from practice.default.source

In [0]:
%sql
-- Data at target table
select * from practice.default.target

In [0]:
%sql
merge into practice.default.target as t
using 
(
  with target as 
(
    select * from practice.default.target where valid_flag='y'
),
target_max_key as 
(
  select max_key from practice.default.config_table where table_name="practice.default.target"
)
select max_key+row_number() over(order by (select null)) as join_id,s.id,s.name,s.updated_date as effective_date,"2099-01-01" as end_date,"y" as valid_flag
from practice.default.source as s
left join 
target as t 
on s.id=t.id
join 
target_max_key
where s.name<>t.name or t.id is null
union 
select t.id,t.id,t.name,t.effective_date,to_date(date_add(day,-1,s.updated_date)) as end_date,"n"
from practice.default.source as s
inner join 
target as t 
on s.id=t.id
where s.name<>t.name
) s
on s.join_id=t.id and t.valid_flag='y'
when matched then 
update 
set t.end_date=s.end_date,
t.valid_flag=s.valid_flag
when not matched then
insert(t.sk,t.id,t.name,t.effective_date,t.end_date,t.valid_flag)
values(s.join_id,s.id,s.name,s.effective_date,s.end_date,s.valid_flag)

In [0]:
%sql
update practice.default.config_table
set max_key=(select max(sk) from practice.default.target)
where table_name='practice.default.target'

In [0]:
%sql
select * from practice.default.target